In [ ]:
import pandas as pd
import pypsa
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

### Analyse the cost structure of the full steel model vs HBI+EAF split

#### Read networks

In [ ]:
techs = ["eaf", "eaf-grid", "hbi", "steel", "steel-ploud90"]

In [ ]:
n_eaf = pypsa.Network("../../resources/lco-eaf/cost_year~2030/South_South_America/network_1.nc")
n_eaf_grid = pypsa.Network("../../resources/lco-eaf-grid/cost_year~2030/South_South_America/network_1.nc")
n_hbi = pypsa.Network("../../resources/lco-hbi/cost_year~2030/South_South_America/network_1.nc")
n_steel = pypsa.Network("../../resources/lco-steel/cost_year~2030/South_South_America/network_1.nc")
n_steel_pload90 = pypsa.Network("../../resources/lco-steel/cost_year~2030/South_South_America/network_1-90partload.nc")

In [ ]:
costs_per_carrier_hbi = n_hbi.statistics.system_cost() / n_hbi.statistics.energy_balance().loc["Load","HBI","HBI"]
costs_per_carrier_hbi #.sum()

In [ ]:
costs_per_carrier_eaf = n_eaf.statistics.system_cost() / n_eaf.statistics.energy_balance().loc["Load","Steel","Steel"]
costs_per_carrier_eaf #.sum()

In [ ]:
costs_per_carrier_eaf_grid = n_eaf_grid.statistics.system_cost() / n_eaf_grid.statistics.energy_balance().loc["Load","Steel","Steel"]
costs_per_carrier_eaf_grid #.sum()

In [ ]:
costs_per_carrier_steel = n_steel.statistics.system_cost() / n_steel.statistics.energy_balance().loc["Load","Steel","Steel"]
costs_per_carrier_steel #.sum()

In [ ]:
costs_per_carrier_steel_pload90 = n_steel_pload90.statistics.system_cost() / n_steel_pload90.statistics.energy_balance().loc["Load","Steel","Steel"]
costs_per_carrier_steel_pload90 #.sum()

In [ ]:
# Stacked bar chart of costs per carrier for single region — show three tech bars stacked by carrier
# Flatten and group carriers, convert to positive costs
def flatten_and_group(s):
    idx = [c[-1] if isinstance(c, tuple) else c for c in s.index]
    s2 = s.copy()
    s2.index = idx
    return s2.groupby(level=0).sum() * (-1)

costs_hbi = flatten_and_group(costs_per_carrier_hbi)
costs_eaf = flatten_and_group(costs_per_carrier_eaf)
costs_eaf_grid = flatten_and_group(costs_per_carrier_eaf_grid)
costs_steel = flatten_and_group(costs_per_carrier_steel)
costs_steel_pload90 = flatten_and_group(costs_per_carrier_steel_pload90)
    

# Combine into a single DataFrame: rows = carriers, cols = techs
costs_df = pd.DataFrame({
    'EAF': costs_eaf,
    'EAF_grid': costs_eaf_grid,
    'HBI': costs_hbi,
    'Steel': costs_steel,
    'Steel_pload90': costs_steel_pload90
}).fillna(0)

# Ensure deterministic carrier order
carriers = costs_df.index.tolist()

# Plot stacked bars: use DataFrame plotting (transpose so columns=carriers -> stacked)
fig, ax = plt.subplots(figsize=(10, 4))
colors = sns.color_palette('Set2', len(carriers))
# transpose so index is tech, columns are carriers; stacked by carriers
costs_df.T.plot(kind='bar', stacked=True, ax=ax, color=colors, edgecolor='none')

ax.set_ylabel('Cost (€/t_steel)')
ax.set_xlabel('Technology')
ax.set_xticklabels(['EAF', 'EAF-grid', 'HBI', 'Steel', 'Steel_pload90'], rotation=0)
ax.set_title('Steel supply costs per carrier (single region)')
ax.grid(axis='y', alpha=0.4, zorder=0)
ax.legend(title='Carrier', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_ylim(0, None)
plt.tight_layout()
plt.show()

### Misc

In [ ]:
n = n_steel

In [ ]:
# Electricity to steel ratio
steel_in_Mt = n.statistics.energy_balance().loc["Load", :,"Steel"].sum() /1e6 * (-1) # from t to Mt
steel_in_Mt

In [ ]:
# Electricity to steel ratio
electricity_in_TWh = n.statistics.energy_balance().loc["Generator", ["Photovoltaics", "Wind energy"],:].sum() / 1e6 # from MWh to TWh

electricity_in_TWh

In [ ]:
ratio = electricity_in_TWh / steel_in_Mt
print(f"Electricity to steel ratio: {ratio:.2f} TWh/Mt")

In [ ]:
n.carriers.loc["hydrogen", "color"] = "blue"

In [ ]:
n.statistics.energy_balance() #.iplot()